# H3: LLM serving on Kaggle (2 x T4)

Settings: Accelerator **GPU T4 x2**, Internet **on**, secret **HF_TOKEN** (Add-ons -> Secrets).
Licences accepted on Hugging Face for the token's account: Gemma 2, Llama 3.1, Mistral-7B-Instruct-v0.3.

Cells 1-4 set up every session. Cell 5 is the serving check (E08-R008) that pins the models (Q-D14).
Cell 6 packs the results for download. Cell 7 shows how an experiment script uses the stack.
See `kaggle/README.md`.

In [ ]:
# 1. Hugging Face token
import os
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

In [ ]:
# 2. Code
!git clone -q -b claude/project-thread-32dk2s https://github.com/santoshcheethiralame-dot/MissingMass /kaggle/working/MissingMass
%cd /kaggle/working/MissingMass
!git log --oneline -1

In [ ]:
# 3. Install (about 5 min). vLLM brings its own torch (2.10, CUDA 12.8); version pinned in configs/llm/models.json.
!pip install -q "vllm==0.19.1" bitsandbytes
!pip install -q -e . --no-deps
import vllm, torch; print("vllm", vllm.__version__, "torch", torch.__version__, "cuda", torch.version.cuda)

In [ ]:
# 4. GPUs
!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv

In [ ]:
# 5. Serving check (E08-R008): about 1 hour. Prints one line per model; resumes if re-run.
!python analysis/e08/serving_check.py --out results/E08/E08-R008 --group pilots --pair mistral-7b,qwen2.5-7b --record

In [ ]:
# 6. Pack the results for download (Output panel -> E08-R008.zip)
!cd results/E08 && zip -qr /kaggle/working/E08-R008.zip E08-R008 && ls -la /kaggle/working/E08-R008.zip

In [ ]:
# 7. Using the stack in an experiment script (example; not part of E08-R008)
# from crsafety.llm import CallCache, models
# from crsafety.llm.serve import Server
# os.environ["CRSAFETY_LLM_CACHE"] = "/kaggle/working/cache/llm"          # back this up as a Kaggle dataset
# os.environ["CRSAFETY_LLM_CACHE_READ"] = "/kaggle/input/<your-cache-dataset>/llm"  # earlier sessions
# with Server(models.resolve("mistral-7b", require_pinned=True), gpu=0, port=8000) as tgt, \
#      Server(models.resolve("qwen2.5-7b", require_pinned=True), gpu=1, port=8001) as aud:
#     t, a = tgt.client(CallCache()), aud.client(CallCache())
#     r = a.chat([{"role": "user", "content": "hi"}], seed=1, temperature=0.7, max_tokens=64)